In [4]:
%pip install langchain langchain-community langchain-core langchain-text-splitters langchain-huggingface langchain-openai langchain-classic pypdf chromadb sentence-transformers ipykernel

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import AzureChatOpenAI
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
from langchain_openai import ChatOpenAI

# --- 1. PwC Gateway Configuration ---
# Replace with your actual strings
PWC_GATEWAY_URL = "https://your-pwc-internal-base-url.pwcinternal.com/v1" # Ensure it ends with /v1 if required by the gateway
PWC_API_KEY = "your_pwc_api_key_here"

llm = ChatOpenAI(
    base_url=PWC_GATEWAY_URL,
    api_key=PWC_API_KEY,
    model="azure-gpt-4.1-mini",  # The proxy maps this to the correct backend instance
    temperature=0.2
)

# --- 2. Load the Local Database ---
CHROMA_DB_DIR = "./chroma_db"
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_db = Chroma(persist_directory=CHROMA_DB_DIR, embedding_function=embedding_model)
retriever = vector_db.as_retriever(search_kwargs={"k": 3})

# --- 3. Define the Prompt Structure ---
system_prompt = (
    "You are a helpful and precise medical assistant. Use the following retrieved context "
    "from the Ozempic prescribing label to answer the user's question. If you don't know the answer "
    "based on the context, just say that you don't know. Do not make up information.\n\n"
    "Context: {context}"
)
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

# --- 4. Build the RAG Chain ---
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

print("✅ System successfully configured for the PwC Gateway Proxy!")

In [ ]:
def ask_ozempic_bot(question):
    print(f"Searching for: '{question}'...")
    
    # This triggers the search AND sends it to Azure
    response = rag_chain.invoke({"input": question})
    
    print("\n" + "-"*50)
    print("🤖 BOT ANSWER:")
    print("-"*50)
    print(response['answer'])
    print("\n(Source Document Chunks Used):")
    for i, doc in enumerate(response['context']):
        print(f" - Chunk {i+1} (Page {doc.metadata.get('page', 'Unknown')})")

In [ ]:
ask_ozempic_bot("What should I do if I miss a dose of my medication?")